In [13]:
import numpy as np
import matplotlib.pyplot as plt

In [14]:
def bosong_0(x):
    global sigma_0, k_0
    psi_0 = np.exp(-1/2 * ((x-5)/sigma_0)**2 )*np.exp(1j*k_0*x) * (np.pi*sigma_0**2)**(-1/4)
    return psi_0

def thenang(x):
    if x <= 0 or x >= 15:
        V = 0
    else:
        V = 0
    return V

def khoitao_x_t():
    global N_x, N_t, x_min, x_max, t_min, t_max
    x = np.linspace(x_min, x_max, N_x)
    t = np.linspace(t_min, t_max, N_t)
    dx = (x_max - x_min) / (N_x - 1)
    dt = (t_max - t_min) / (N_t - 1)
    return x, t, dx, dt

def create_matrix_M():
    global N_x, N_t, x_min, x_max, t_min, t_max, dx, dt
    g = np.zeros(N_x, dtype=complex)
    a = np.zeros(N_x, dtype=complex)
    for i in range(N_x):
        x_i = x_min + i*dx
        V = thenang(x_i)
        g[i] = -5j/(4*dx**2) - 1j*V
        a[i] = 1j/(24*dx**2)
    diag = np.diag(g)
    off_diag1 = np.diag([16*a[i]]*(N_x-1), 1) + np.diag([16*a[i]]*(N_x-1), -1)
    off_diag2 = np.diag([-a[i]]*(N_x-2), 2) + np.diag([-a[i]]*(N_x-2), -2)
    M = diag + off_diag1 + off_diag2
    return M

def giaipt_psi_RK4():
    global N_x, N_t, x_min, x_max, t_min, t_max, dx, dt
    y = np.zeros([N_t,N_x], dtype=np.complex128)
    x, t, dx, dt = khoitao_x_t()
    M = create_matrix_M()
    psi_0 = bosong_0(x)
    y[0] = psi_0
    for i in range(0, N_t-1):
        k1 = np.dot(M, y[i])
        k2 = np.dot(M, y[i] + 0.5*dt*k1)
        k3 = np.dot(M, y[i] + 0.5*dt*k2)
        k4 = np.dot(M, y[i] + dt*k3)
        y[i+1] = y[i] + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

        y[i+1, 0] = 0
        y[i+1, -1] = 0
    return y

In [27]:
def luu_ketqua_psi(output_name, x, t, y, save_every=100):
    filename = f"TDSE-ketqua-{output_name}.txt"

    with open(filename, "w", encoding="utf-8") as f:
        f.write("# KET QUA GIAI PHUONG TRINH SCHRODINGER PHU THUOC THOI GIAN\n\n")

        f.write("#" * 150 + "\n")
        f.write("# THAM SO DAU VAO\n")
        f.write("#" * 150 + "\n\n")

        f.write(f"# N_x        = {N_x}\n")
        f.write(f"# N_t        = {N_t}\n")
        f.write(f"# x_min      = {x_min}\n")
        f.write(f"# x_max      = {x_max}\n")
        f.write(f"# t_min      = {t_min}\n")
        f.write(f"# t_max      = {t_max}\n")
        f.write(f"# dx         = {dx}\n")
        f.write(f"# dt         = {dt}\n")
        f.write(f"# sigma_0    = {sigma_0}\n")
        f.write(f"# k_0        = {k_0}\n")
        f.write(f"# save_every = {save_every}\n")

        f.write("\n" + "#" * 150 + "\n\n")

        f.write(f"#{'t':>15s} {'x':>15s} {'Re_psi':>15s} {'Im_psi':>15s} {'|psi|':>15s} {'|psi|^2':>15s}\n")

        for n in range(0, len(t), save_every):
            for i in range(len(x)):
                psi = y[n, i]

                f.write(f" {t[n]:>15.6e} "
                        f"{x[i]:>15.6e} "
                        f"{psi.real:>15.6e} "
                        f"{psi.imag:>15.6e} "
                        f"{np.abs(psi):>15.6e} "
                        f"{np.abs(psi)**2:>15.6e}\n")

            f.write("\n")

In [38]:
def luu_file_xac_xuat(output_name, x, t, y, save_every=100):
    filename = f"TDSE-ketqua-{output_name}-xacxuat.txt"

    with open(filename, "w", encoding="utf-8") as f:
        f.write("# KET QUA GIAI PHUONG TRINH SCHRODINGER PHU THUOC THOI GIAN\n\n")

        f.write("#" * 150 + "\n")
        f.write("# THAM SO DAU VAO\n")
        f.write("#" * 150 + "\n\n")

        f.write(f"# N_x        = {N_x}\n")
        f.write(f"# N_t        = {N_t}\n")
        f.write(f"# x_min      = {x_min}\n")
        f.write(f"# x_max      = {x_max}\n")
        f.write(f"# t_min      = {t_min}\n")
        f.write(f"# t_max      = {t_max}\n")
        f.write(f"# dx         = {dx}\n")
        f.write(f"# dt         = {dt}\n")
        f.write(f"# sigma_0    = {sigma_0}\n")
        f.write(f"# k_0        = {k_0}\n")
        f.write(f"# save_every = {save_every}\n")

        f.write("\n" + "#" * 150 + "\n\n")

        f.write(f"#{'t':>15s} {'Integral_|psi|^2_dx':>25s}\n")

        for n in range(0, len(t), save_every):

            prob_x = np.abs(y[n, :])**2
            tong_xac_xuat = np.trapz(prob_x, x)

            f.write(f" {t[n]:>15.6e} "
                    f"{tong_xac_xuat:>25.6e}\n")

In [50]:
sigma_0 = 0.5
k_0 = 8.0

N_x = 401
N_t = 6001
x_min = 0.0
x_max = 15.0
t_min = 0.0
t_max = 1.5

In [51]:
x, t, dx, dt = khoitao_x_t()

y = giaipt_psi_RK4()

In [52]:
luu_ketqua_psi("test", x, t, y, save_every=200)

In [53]:
luu_file_xac_xuat("test", x, t, y, save_every=100)

In [55]:
dt

0.00025